In [ ]:
!nvidia-smi

Wed Sep  2 07:12:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q unsloth
!pip install -q sentence-transformers faiss-cpu pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

In [ ]:
print("All required libraries installed successfully.")

All required libraries installed successfully.


In [ ]:
import os
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel

print("Libraries imported successfully.")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Libraries imported successfully.


In [ ]:
documents = [
    """
    Artificial Intelligence, or AI, is a branch of computer science
    that focuses on creating systems capable of performing tasks that
    normally require human intelligence. These tasks include learning,
    reasoning, problem solving, decision making and understanding language.
    """,

    """
    Machine Learning is a subset of Artificial Intelligence.
    It enables computers to learn patterns from data and make predictions
    or decisions without being explicitly programmed for every task.
    """,

    """
    Deep Learning is a specialized area of Machine Learning that uses
    artificial neural networks containing multiple layers. Deep learning
    is commonly used in computer vision, natural language processing,
    speech recognition and other AI applications.
    """,

    """
    Large Language Models, commonly called LLMs, are artificial intelligence
    models trained on very large collections of text. LLMs can perform
    tasks such as question answering, summarization, translation,
    classification and text generation.
    """,

    """
    Retrieval-Augmented Generation, or RAG, combines information retrieval
    with language generation. A retrieval system first searches a knowledge
    base for information relevant to a user's question. The retrieved
    information is then provided to a language model as context so that
    the model can generate a more grounded answer.
    """
]

print("Number of documents:", len(documents))

Number of documents: 5


In [ ]:
for i, document in enumerate(documents):

    print("\n" + "=" * 70)
    print("DOCUMENT", i + 1)
    print("=" * 70)

    print(document)


DOCUMENT 1

    Artificial Intelligence, or AI, is a branch of computer science
    that focuses on creating systems capable of performing tasks that
    normally require human intelligence. These tasks include learning,
    reasoning, problem solving, decision making and understanding language.
    

DOCUMENT 2

    Machine Learning is a subset of Artificial Intelligence.
    It enables computers to learn patterns from data and make predictions
    or decisions without being explicitly programmed for every task.
    

DOCUMENT 3

    Deep Learning is a specialized area of Machine Learning that uses
    artificial neural networks containing multiple layers. Deep learning
    is commonly used in computer vision, natural language processing,
    speech recognition and other AI applications.
    

DOCUMENT 4

    Large Language Models, commonly called LLMs, are artificial intelligence
    models trained on very large collections of text. LLMs can perform
    tasks such as question answer

In [ ]:
def chunk_text(text, chunk_size=80):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        chunks.append(chunk)

    return chunks


chunks = []

for document in documents:

    chunks.extend(
        chunk_text(document)
    )


print("Total chunks:", len(chunks))

Total chunks: 5


In [ ]:
for i, chunk in enumerate(chunks):

    print("\n" + "=" * 70)
    print("CHUNK", i + 1)
    print("=" * 70)

    print(chunk)


CHUNK 1
Artificial Intelligence, or AI, is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem solving, decision making and understanding language.

CHUNK 2
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

CHUNK 3
Deep Learning is a specialized area of Machine Learning that uses artificial neural networks containing multiple layers. Deep learning is commonly used in computer vision, natural language processing, speech recognition and other AI applications.

CHUNK 4
Large Language Models, commonly called LLMs, are artificial intelligence models trained on very large collections of text. LLMs can perform tasks such as question answering, summarization, translation, classification and text generation.

CHUNK 5
Retrieval

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (5, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings.astype("float32")
)

print("Vectors stored:", index.ntotal)

Vectors stored: 5


In [ ]:
def retrieve_documents(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({
            "text": chunks[idx],
            "score": float(score)
        })

    return results

In [ ]:
query = "What is Retrieval-Augmented Generation?"

results = retrieve_documents(
    query,
    top_k=3
)

for i, result in enumerate(results):

    print("\n" + "=" * 70)
    print("RETRIEVED RESULT", i + 1)
    print("=" * 70)

    print("Similarity Score:")
    print(result["score"])

    print("\nText:")
    print(result["text"])


RETRIEVED RESULT 1
Similarity Score:
0.7140891551971436

Text:
Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation. A retrieval system first searches a knowledge base for information relevant to a user's question. The retrieved information is then provided to a language model as context so that the model can generate a more grounded answer.

RETRIEVED RESULT 2
Similarity Score:
0.21725383400917053

Text:
Large Language Models, commonly called LLMs, are artificial intelligence models trained on very large collections of text. LLMs can perform tasks such as question answering, summarization, translation, classification and text generation.

RETRIEVED RESULT 3
Similarity Score:
0.15986105799674988

Text:
Machine Learning is a subset of Artificial Intelligence. It enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.


In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None
)

print("4-bit quantized model loaded successfully.")

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit quantized model loaded successfully.


In [ ]:
FastLanguageModel.for_inference(model)

print("Inference mode enabled.")

Inference mode enabled.


In [ ]:
prompt = """
Explain artificial intelligence in simple words.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

generated_tokens = outputs[
    0
][
    inputs["input_ids"].shape[1]:
]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("MODEL RESPONSE:")
print(answer)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines or software programs that can perform tasks that typically require human intelligence. These tasks include understanding natural language, recognizing patterns and objects in images, playing games, and making decisions based on data. AI uses algorithms and statistical models to learn from data and improve their performance over time. It has applications in fields such as healthcare, finance, transportation, and many others where intelligent decision-making is needed. The goal of AI is to


In [ ]:
def generate_answer(
    question,
    top_k=3
):

    # ------------------------------------------------
    # 1. Retrieve relevant documents
    # ------------------------------------------------

    retrieved = retrieve_documents(
        question,
        top_k=top_k
    )

    # ------------------------------------------------
    # 2. Combine retrieved context
    # ------------------------------------------------

    context = "\n\n".join(
        [
            item["text"]
            for item in retrieved
        ]
    )

    # ------------------------------------------------
    # 3. Create grounded prompt
    # ------------------------------------------------

    prompt = f"""
You are a helpful AI assistant.

Answer the user's question using the provided context.

IMPORTANT:
Use only the information provided in the context.
Do not invent facts.

If the answer is not available in the context,
say:

"The information is not available in the provided documents."

CONTEXT:

{context}

QUESTION:

{question}

ANSWER:
"""

    # ------------------------------------------------
    # 4. Tokenize
    # ------------------------------------------------

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    # ------------------------------------------------
    # 5. Generate answer
    # ------------------------------------------------

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    # ------------------------------------------------
    # 6. Remove prompt from output
    # ------------------------------------------------

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    # ------------------------------------------------
    # 7. Decode
    # ------------------------------------------------

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer, retrieved

In [ ]:
question = "What is Retrieval-Augmented Generation?"

answer, retrieved = generate_answer(
    question
)

print("=" * 70)
print("USER QUESTION")
print("=" * 70)

print(question)

print("\n" + "=" * 70)
print("GENERATED ANSWER")
print("=" * 70)

print(answer)

print("\n" + "=" * 70)
print("RETRIEVED DOCUMENTS")
print("=" * 70)

for i, item in enumerate(retrieved):

    print("\nChunk", i + 1)

    print(item["text"])

    print(
        "Similarity:",
        item["score"]
    )

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION
What is Retrieval-Augmented Generation?

GENERATED ANSWER
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language generation. This approach involves two main components: 
1. Information Retrieval System: First, it uses an information retrieval system to search through a knowledge base for relevant information related to a specific query. 
2. Contextualized Language Model: Once the relevant information is found, this information serves as context for a language model, allowing it to generate a more accurate and contextually appropriate response. 

In essence, RAG leverages the strengths of both techniques—information retrieval for finding pertinent data and language modeling for generating coherent and informative responses—to enhance the accuracy and relevance of generated content. This method has been particularly useful in various applications where precise and well-informed answers are required based on contextual data.

RE

In [ ]:
questions = [

    "What is artificial intelligence?",

    "What is machine learning?",

    "What is deep learning?",

    "What are large language models?",

    "What is Retrieval-Augmented Generation?"

]

for question in questions:

    answer, retrieved = generate_answer(
        question
    )

    print("\n")
    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUESTION:
What is artificial intelligence?

ANSWER:
Artificial intelligence (AI) is a broad field that encompasses various subfields such as machine learning and deep learning. It involves the creation of intelligent machines that can perform tasks requiring human-like intelligence, including learning, reasoning, problem-solving, and understanding language. The goal of AI is to develop systems that can mimic or exceed human cognitive abilities through algorithms and computational techniques. This includes developing models and methods for pattern recognition, knowledge representation, automated reasoning, and expert systems. In essence, AI aims to create machines that exhibit behaviors similar to those of humans in terms of perception, cognition, and action.


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUESTION:
What is machine learning?

ANSWER:
Machine learning is a subset of Artificial Intelligence that allows computers to learn patterns from data and make predictions or decisions without explicit programming for each task. Deep learning, which involves artificial neural networks with multiple layers, is a specialized area within machine learning that is particularly useful in fields such as computer vision, natural language processing, and speech recognition. The broader field of Artificial Intelligence encompasses all aspects of creating intelligent machines capable of performing tasks requiring human-like intelligence, including but not limited to learning, reasoning, problem-solving, and understanding language.


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUESTION:
What is deep learning?

ANSWER:
Deep learning is a specialized area of Machine Learning that utilizes artificial neural networks with multiple layers. This technology is widely applied in fields such as computer vision, natural language processing, and speech recognition, among others. The primary goal of deep learning is to enable machines to recognize complex patterns and make accurate predictions or decisions based on data input, without needing explicit programming for each individual task.


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUESTION:
What are large language models?

ANSWER:
Large Language Models (LLMs) are artificial intelligence models trained on very large collections of text. They can perform various tasks including question answering, summarization, translation, classification, and text generation. These models leverage deep learning techniques, which involve artificial neural networks with multiple layers, to process and generate human-like responses based on vast amounts of textual data. Retrieval-Augmented Generation (RAG) is an advanced approach where a retrieval system first identifies relevant information from a knowledge base, providing it as context to a language model, thereby enhancing the accuracy and relevance of generated answers. This method enhances the capabilities of LLMs by integrating external knowledge sources into their decision-making processes.


QUESTION:
What is Retrieval-Augmented Generation?

ANSWER:
Retrieval-Augmented Generation (RAG) is a technique that combines informa

In [ ]:
question = "What is quantum computing?"

answer, retrieved = generate_answer(
    question
)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is quantum computing?

ANSWER:
The information is not available in the provided documents.


In [ ]:
print(
    "GPU Memory Allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU Memory Reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    ),
    "GB"
)

GPU Memory Allocated: 1.22 GB
GPU Memory Reserved: 1.35 GB
